## Package the Tableau Cloud workbook

Run from the dashboards folder after the native workbook build. This optional publish step uses the Tableau Hyper API (`pip install tableauhyperapi`). The resulting TWBX includes the four sample tables; no cloud credentials or local CSV paths are required to open it.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile, ZIP_DEFLATED
import csv
import xml.etree.ElementTree as ET

from tableauhyperapi import HyperProcess, Telemetry, Connection, CreateMode, TableDefinition, TableName, SqlType, Inserter

here = Path.cwd()
results = here.parent / "results"
book = ET.parse(here / "tableau/gameplay.twb")
types = {"integer": SqlType.big_int(), "real": SqlType.double(), "string": SqlType.text()}
package = here / "tableau/gameplay.twbx"
# Fixed-layout comparison pages are native Tableau text zones. They preserve a
# readable snapshot even when a viewer cannot query the workbook's data sources.
with (results / "strategy_cells.csv").open(encoding="utf-8-sig", newline="") as f:
    cells = list(csv.DictReader(f))
def add_snapshot(view, title, unit):
    records = [r for r in cells if r["view"] == view]
    columns = sorted({(int(r["position"]), r["category"]) for r in records})
    decks = sorted({(int(r["deck_order"]), r["deck"], r["group"]) for r in records})
    counts = {(r["deck"], r["category"]): (round(100*float(r["slot_share"]), 1) if view == "Printed card colors" else int(r["count"])) for r in records}
    assert len(decks) == 20 and len(counts) == len(decks) * len(columns)
    dashboard = ET.SubElement(book.find("dashboards"), "dashboard", name="Snapshot | " + view)
    ET.SubElement(dashboard, "style")
    width, height = max(1420, 475 + 110*len(columns)), 870
    ET.SubElement(dashboard, "size", minwidth=str(width), maxwidth=str(width), minheight=str(height), maxheight=str(height))
    zones = ET.SubElement(dashboard, "zones")
    def label(x, y, w, h, label, size=11, ink="#20354A", bg="#FFFFFF", bold=False):
        zone = ET.SubElement(zones, "zone", {"id": str(len(zones)+1), "type":"text", "x":str(round(x*100000/width)), "y":str(round(y*100000/height)), "w":str(round(w*100000/width)), "h":str(round(h*100000/height))})
        fmt = ET.SubElement(zone, "formatted-text")
        ET.SubElement(fmt, "run", {"fontname":"Tableau Book", "fontsize":str(size), "fontcolor":ink, "bold":str(bold).lower()}).text = str(label)
        style = ET.SubElement(zone,"zone-style")
        ET.SubElement(style,"format",attr="background-color",value=bg)
        ET.SubElement(style,"format",attr="margin",value="4")
    label(20, 14, 1300, 43, title, 22, bold=True)
    label(20, 62, 1340, 34, unit + "  |  1 player deck and 19 reference lists", 12)
    left, gap = 457, (width - 475) / len(columns)
    label(20, 110, 310, 48, "Deck", 12, bold=True)
    label(330, 110, 125, 48, "Group", 12, bold=True)
    for j, (_, name) in enumerate(columns):
        label(left + j*gap, 110, gap-2, 48, name, 10, bold=True)
    peak = max(counts.values()) or 1
    palette = ["#F2F6F6", "#D9ECE9", "#ADDBD4", "#74B9B1", "#3D918C", "#155E60"]
    for i, (_, deck, group) in enumerate(decks):
        y = 163 + 33*i
        label(20, y, 310, 31, deck, 10, bold=(i==0))
        label(330, y, 125, 31, group, 10)
        for j, (_, name) in enumerate(columns):
            value = counts[deck, name]
            step = round(value*(len(palette)-1)/peak)
            label(left + j*gap, y, gap-2, 31, (f"{value}%" if view in {"Printed card colors", "Casting-cost emphasis"} else value), 12, "#FFFFFF" if step>=4 else "#173638", palette[step], True)
    label(20, 833, 1350, 30, "Darker cells represent higher values within this chart. Fixed snapshot of the workbook data.", 10)
for sheet in book.findall("./worksheets/worksheet"):
    name = sheet.get("name", "")
    if name.startswith("Compare | ") and name.endswith(" marks"):
        view = name[len("Compare | "):-len(" marks")]
        title = "Printed mana curve" if view == "Printed curve" else view
        unit = sheet.find("layout-options/title/formatted-text").findall("run")[1].text.strip(" |")
        if view == "Printed curve":
            unit = "Nonland library slots by mana value; commanders excluded"
        elif view == "Printed card colors":
            unit = "Percent of nonland slots, including the commander; multicolor cards count separately"
        add_snapshot(view, title, unit)
with TemporaryDirectory() as folder:
    with HyperProcess(Telemetry.DO_NOT_SEND_USAGE_DATA_TO_TABLEAU, parameters={"log_dir":folder, "use_tcp_port":"auto"}) as engine:
        with ZipFile(Path(folder) / package.name, "w", compression=ZIP_DEFLATED) as bundle:
            for ds in book.findall("./datasources/datasource"):
                connection = ds.find("connection")
                csv_name = connection.attrib["filename"]
                fields = [(v.attrib["name"].strip("[]"), v.attrib["datatype"]) for v in ds.findall("column")]
                hyper_name = Path(csv_name).with_suffix(".hyper").name
                destination = "Data/Extracts/" + hyper_name
                path = Path(folder) / hyper_name
                table = TableDefinition(TableName("Extract", "Extract"), [TableDefinition.Column(k, types[t]) for k,t in fields])
                with Connection(engine.endpoint, str(path), CreateMode.CREATE_AND_REPLACE) as db:
                    db.catalog.create_schema("Extract")
                    db.catalog.create_table(table)
                    with (results / csv_name).open(encoding="utf-8-sig", newline="") as f, Inserter(db, table) as insert:
                        rows = list(csv.DictReader(f))
                        for row in rows:
                            insert.add_row([row[k] if t == "string" else None if row[k] == "" else int(row[k]) if t == "integer" else float(row[k]) for k,t in fields])
                        insert.execute()
                    assert db.execute_scalar_query('SELECT COUNT(*) FROM "Extract"."Extract"') == len(rows)
                bundle.write(path, destination)
                connection.clear()
                connection.set("class", "federated")
                key = "hyper." + ds.attrib["name"]
                named = ET.SubElement(ET.SubElement(connection, "named-connections"), "named-connection", {"name":key, "caption":csv_name})
                ET.SubElement(named, "connection", {"class":"hyper", "dbname":destination, "schema":"Extract", "tablename":"Extract", "access_mode":"readonly"})
                ET.SubElement(connection, "relation", {"name":"Extract", "table":"[Extract].[Extract]", "connection":key, "type":"table"})
                metadata = ET.SubElement(connection, "metadata-records")
                for ordinal, (k, t) in enumerate(fields):
                    item = ET.SubElement(metadata, "metadata-record", {"class":"column"})
                    for tag, value in [("remote-name",k), ("remote-type",{"string":"129","integer":"20","real":"5"}[t]), ("local-name","["+k+"]"), ("parent-name","[Extract]"), ("remote-alias",k), ("ordinal",str(ordinal)), ("local-type",t), ("aggregation","Sum" if t != "string" else "Count"), ("contains-null","true")]:
                        ET.SubElement(item,tag).text = value
            bundle.writestr("gameplay.twb", ET.tostring(book.getroot(), encoding="utf-8", xml_declaration=True))
    from shutil import copyfile
    copyfile(Path(folder) / package.name, package)
print(package.name, "contains four embedded extracts.")
